# Clean text pipeline

In [1]:

import pandas as pd
import re
import unicodedata
import html
from pathlib import Path


# ============================================================
# 1. CONFIG
# ============================================================

# Full dataset
input_full_path = "labeled_data.csv"
output_full_path = "data_cleaned_full.csv"

# Gold set
gold_path = "human_gold_500_final.csv"
output_gold_path = "human_gold_500_cleaned_final.csv"


# ============================================================
# 2. READ FILE
# ============================================================

def read_table(path):
    path = Path(path)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Không hỗ trợ định dạng file: {path.suffix}")


# ============================================================
# 3. TEXT NORMALIZATION / CLEANING
# ============================================================

def normalize_unicode(text):
    return unicodedata.normalize("NFC", str(text))


def normalize_for_match(text):
    """
    Chỉ dùng để match gold với full.
    Không dùng để thay thế nội dung text gốc.
    """
    if pd.isna(text):
        return ""
    text = normalize_unicode(str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def replace_emoji_and_emoticon(text):
    """
    Không xóa emoji hoàn toàn.
    Chuyển emoji/emoticon phổ biến thành token tiếng Việt để giữ tín hiệu cảm xúc.
    """
    replacements = {
        # Laugh / sarcasm / joke
        "😂": " haha ",
        "🤣": " haha ",
        "😅": " haha ",
        "😆": " haha ",
        "😁": " haha ",
        "😀": " haha ",
        "😄": " haha ",
        "=)))": " haha ",
        "=))": " haha ",
        ":)))": " haha ",
        ":))": " haha ",
        "kkk": " haha ",
        "kk": " haha ",
        "kakaka": " haha ",
        "kaka": " haha ",

        # Positive / love
        "❤️": " trái tim ",
        "❤": " trái tim ",
        "😍": " yêu thích ",
        "🥰": " yêu thích ",
        "😘": " yêu thích ",
        "👍": " thích ",
        "🔥": " cháy ",
        "🎉": " chúc mừng ",
        "🥳": " chúc mừng ",

        # Negative / angry / sad
        "👎": " không thích ",
        "😡": " tức giận ",
        "🤬": " tức giận ",
        "😭": " buồn ",
        "😢": " buồn ",
        "🥲": " buồn ",

        # Neutral / unclear
        "😶": " biểu cảm trung tính ",
        "🙂": " biểu cảm trung tính ",
        "🙃": " biểu cảm trung tính ",
        "😌": " biểu cảm trung tính ",

        # Topic-specific emoji
        "🔔": " rung chuông ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def reduce_repeated_chars(text):
    """
    Rút gọn ký tự chữ lặp quá dài.
    Ví dụ: đẹppppp -> đẹpp, siuuuuuu -> siuu.
    Không áp dụng cho chữ số để tránh làm sai giá/điểm số.
    """
    vietnamese_letters = (
        r"a-zA-Z"
        r"àáạảãâầấậẩẫăằắặẳẵ"
        r"èéẹẻẽêềếệểễ"
        r"ìíịỉĩ"
        r"òóọỏõôồốộổỗơờớợởỡ"
        r"ùúụủũưừứựửữ"
        r"ỳýỵỷỹđ"
        r"ÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴ"
        r"ÈÉẸẺẼÊỀẾỆỂỄ"
        r"ÌÍỊỈĨ"
        r"ÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠ"
        r"ÙÚỤỦŨƯỪỨỰỬỮ"
        r"ỲÝỴỶỸĐ"
    )

    pattern = rf"([{vietnamese_letters}])\1{{2,}}"
    return re.sub(pattern, r"\1\1", text)


def basic_replacements(text):
    """
    Chuẩn hóa teencode/viết tắt phổ biến.
    Chỉ thay các mẫu tương đối chắc chắn.
    """
    replacements = [
        (r"\bko\b", "không"),
        (r"\bk\b", "không"),
        (r"\bkhum\b", "không"),
        (r"\bhok\b", "không"),
        (r"\bhong\b", "không"),
        (r"\bkh\b", "không"),
        (r"\bđc\b", "được"),
        (r"\bdc\b", "được"),
        (r"\bđg\b", "đang"),
        (r"\bmn\b", "mọi người"),
        (r"\bvs\b", "với"),
        (r"\bck\b", "chung kết"),
        (r"\bclb\b", "câu lạc bộ"),
        (r"\bc1\b", "uefa champions league"),
        (r"\bblv\b", "bình luận viên"),
    ]

    for pattern, repl in replacements:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)

    # Chuẩn hóa số tiền: 25tr / 25 củ -> 25 triệu
    text = re.sub(r"\b(\d+)\s*tr\b", r"\1 triệu", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(\d+)\s*củ\b", r"\1 triệu", text, flags=re.IGNORECASE)

    return text


def topic_specific_replacements(text, topic):
    """
    Chuẩn hóa từ khóa theo từng topic.
    Có topic đi kèm để tránh thay sai ngữ cảnh.
    """
    topic = str(topic).lower()
    t = text

    if topic == "arsenal":
        replacements = [
            (r"\bars\b", "arsenal"),
            (r"\basn\b", "arsenal"),
            (r"\bthe gunners\b", "arsenal"),
            (r"\bpháo thủ\b", "arsenal pháo thủ"),
            (r"\bpháo\b", "arsenal pháo"),
            (r"\bphấn đào\b", "arsenal phấn đào"),
            (r"\bphấn\b", "arsenal phấn"),
            (r"\ba sơ nan\b", "arsenal a sơ nan"),
            (r"\bsơ nan\b", "arsenal sơ nan"),
            (r"\bphốn lào\b", "arsenal phốn lào"),
            (r"\bphú ngao\b", "arsenal phú ngao"),
            (r"\ba phú thảo\b", "arsenal a phú thảo"),
            (r"\betita\b", "arteta"),
            (r"\bateta\b", "arteta"),
            (r"\baterta\b", "arteta"),
            (r"\bmc\b", "manchester city"),
            (r"\bmct\b", "manchester city"),
            (r"\bman xanh\b", "manchester city"),
            (r"\bwh\b", "west ham"),
        ]

    elif topic == "iphone17series":
        replacements = [
            (r"\bip air\b", "iphone air"),
            (r"\bip17\b", "iphone 17"),
            (r"\bip 17\b", "iphone 17"),
            (r"\bip\b", "iphone"),
            (r"\b17prm\b", "iphone 17 pro max"),
            (r"\b17 prm\b", "iphone 17 pro max"),
            (r"\b17 promax\b", "iphone 17 pro max"),
            (r"\b17 pro max\b", "iphone 17 pro max"),
            (r"\b17 pro\b", "iphone 17 pro"),
            (r"\b16prm\b", "iphone 16 pro max"),
            (r"\b16 prm\b", "iphone 16 pro max"),
            (r"\b16 promax\b", "iphone 16 pro max"),
            (r"\b16 pro max\b", "iphone 16 pro max"),
            (r"\b16pro\b", "iphone 16 pro"),
            (r"\b16 pro\b", "iphone 16 pro"),
        ]

    elif topic == "s25series":
        replacements = [
            (r"\bs25u\b", "samsung galaxy s25 ultra"),
            (r"\bs25 ultra\b", "samsung galaxy s25 ultra"),
            (r"\bs25 plus\b", "samsung galaxy s25 plus"),
            (r"\bs25\+\b", "samsung galaxy s25 plus"),
            (r"\bs25\b", "samsung galaxy s25"),
            (r"\bss\b", "samsung"),
            (r"\bsam\b", "samsung"),
            (r"\bsnap\b", "snapdragon"),
            (r"\bexy\b", "exynos"),
        ]

    elif topic == "ronaldo":
        replacements = [
            (r"\bcr7\b", "ronaldo cr7"),
            (r"\bcristiano\b", "ronaldo cristiano"),
            (r"\brô\b", "ronaldo"),
            (r"\bsiu+\b", "siuu ronaldo"),
            (r"\bsiuuu+\b", "siuu ronaldo"),
            (r"\banh 7\b", "ronaldo anh 7"),
            (r"\banh bảy\b", "ronaldo anh bảy"),
        ]

    elif topic == "vinfast_vf3":
        replacements = [
            (r"\bvf 3\b", "vinfast vf3"),
            (r"\bvf3\b", "vinfast vf3"),
            (r"\bvinfast vf3\b", "vinfast vf3"),
        ]

    else:
        replacements = []

    for pattern, repl in replacements:
        t = re.sub(pattern, repl, t, flags=re.IGNORECASE)

    return t


def clean_comment(text, topic=""):
    """
    Hàm clean chính.
    Luôn tạo clean_text mới từ cột text gốc.
    Không dùng clean_text cũ làm nguồn.
    Không chuẩn hóa nhãn ở bước này.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    text = html.unescape(text)
    text = normalize_unicode(text)

    # Xóa ký tự điều khiển, zero-width
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)

    # Chuyển emoji/emoticon trước khi lowercase
    text = replace_emoji_and_emoticon(text)

    # URL, mention, hashtag
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"@\w+", " <user> ", text)
    text = re.sub(r"#", " ", text)

    # Timestamp kiểu 0:21, 1:11
    text = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", " <time> ", text)

    # Lowercase
    text = text.lower()

    # Chuẩn hóa cơ bản
    text = basic_replacements(text)

    # Chuẩn hóa theo topic
    text = topic_specific_replacements(text, topic)

    # Bỏ ký tự trang trí, vẫn giữ ? ! vì có ý nghĩa câu hỏi/cảm xúc
    text = re.sub(r"[►•▪●▶]", " ", text)

    # Rút gọn ký tự lặp
    text = reduce_repeated_chars(text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    # Unicode lần cuối
    text = normalize_unicode(text)

    return text


# ============================================================
# 4. MATCH GOLD WITH FULL WITHOUT USING id
# ============================================================

def ensure_match_columns(frame):
    """
    Tạo cột khóa phụ để kiểm tra gold có match đúng full không.
    Không dùng id đơn lẻ.
    """
    frame = frame.copy()

    for col in ["topic", "video_id", "source_item_id", "created_at", "text", "clean_text"]:
        if col not in frame.columns:
            frame[col] = ""

    frame["__topic_key"] = frame["topic"].apply(normalize_for_match)
    frame["__video_id_key"] = frame["video_id"].apply(normalize_for_match)
    frame["__source_item_id_key"] = frame["source_item_id"].apply(normalize_for_match)
    frame["__created_at_key"] = frame["created_at"].apply(normalize_for_match)
    frame["__text_key"] = frame["text"].apply(normalize_for_match)
    frame["__clean_text_key"] = frame["clean_text"].apply(normalize_for_match)

    return frame


def build_match_key(row, fields):
    return "||".join(str(row.get(f"__{field}_key", "")) for field in fields)


def valid_match_key(key, fields):
    parts = key.split("||")

    if len(parts) != len(fields):
        return False

    for field, value in zip(fields, parts):
        if field in {"topic", "text", "clean_text"} and value == "":
            return False

    for field, value in zip(fields, parts):
        if field in {"video_id", "source_item_id", "created_at"} and value == "":
            return False

    return True


def make_full_key_map(full_work, fields):
    tmp = full_work[["__full_index"] + [f"__{field}_key" for field in fields]].copy()
    tmp["__match_key"] = tmp.apply(lambda row: build_match_key(row, fields), axis=1)
    tmp = tmp[tmp["__match_key"].apply(lambda x: valid_match_key(x, fields))]
    return tmp.groupby("__match_key")["__full_index"].apply(list).to_dict()


def match_gold_to_full(full, gold):
    """
    Chỉ dùng để kiểm tra gold set có match đúng full dataset không.
    Không dùng kết quả này để merge clean_text bằng id.
    """
    full_work = ensure_match_columns(full)
    gold_work = ensure_match_columns(gold)

    full_work["__full_index"] = full_work.index
    gold_work["__gold_row_index"] = gold_work.index

    match_stages = [
        ("topic+video_id+created_at+text", ["topic", "video_id", "created_at", "text"], False),
        ("topic+source_item_id+created_at+text", ["topic", "source_item_id", "created_at", "text"], False),
        ("topic+video_id+text", ["topic", "video_id", "text"], False),
        ("topic+source_item_id+text", ["topic", "source_item_id", "text"], False),
        ("topic+created_at+text", ["topic", "created_at", "text"], False),
        ("topic+video_id+created_at_unique", ["topic", "video_id", "created_at"], True),
        ("topic+source_item_id+created_at_unique", ["topic", "source_item_id", "created_at"], True),
        ("topic+text_unique", ["topic", "text"], True),
        ("topic+clean_text_unique", ["topic", "clean_text"], True),
    ]

    matched_gold_indices = set()
    reports = []

    for method, fields, require_unique in match_stages:
        full_map = make_full_key_map(full_work, fields)

        for _, gold_row in gold_work.iterrows():
            gold_idx = int(gold_row["__gold_row_index"])

            if gold_idx in matched_gold_indices:
                continue

            key = build_match_key(gold_row, fields)

            if not valid_match_key(key, fields):
                continue

            candidates = full_map.get(key, [])

            if not candidates:
                continue

            candidates = sorted(set(int(x) for x in candidates))

            if require_unique and len(candidates) != 1:
                continue

            matched_gold_indices.add(gold_idx)

            reports.append({
                "gold_row_index": gold_idx,
                "gold_topic": gold_row.get("topic", ""),
                "match_method": method,
                "matched_full_count": len(candidates),
                "matched_full_topics": "|".join(str(full.loc[x, "topic"]) for x in candidates),
            })

    for _, gold_row in gold_work.iterrows():
        gold_idx = int(gold_row["__gold_row_index"])

        if gold_idx not in matched_gold_indices:
            reports.append({
                "gold_row_index": gold_idx,
                "gold_topic": gold_row.get("topic", ""),
                "match_method": "unmatched",
                "matched_full_count": 0,
                "matched_full_topics": "",
            })

    return pd.DataFrame(reports).sort_values("gold_row_index").reset_index(drop=True)


# ============================================================
# 5. QUALITY CHECK
# ============================================================

def quality_check(data, name="dataset"):
    print(f"========== QUALITY CHECK: {name} ==========")
    print("Tổng số dòng:", len(data))

    print("Số clean_text trống:",
          data["clean_text"].fillna("").astype(str).str.strip().eq("").sum())

    print("Số clean_text còn URL gốc:",
          data["clean_text"].str.contains(r"http|www\.", case=False, regex=True, na=False).sum())

    print("Số clean_text chưa NFC:",
          data["clean_text"].apply(lambda s: unicodedata.normalize("NFC", str(s)) != str(s)).sum())

    print("Số clean_text còn ký tự chữ lặp >= 3:",
          data["clean_text"].str.contains(r"([a-zA-Zà-ỹÀ-Ỹ])\1{2,}", regex=True, na=False).sum())

    if "topic" in data.columns:
        print("\nPhân phối topic:")
        print(data["topic"].value_counts(dropna=False).sort_index())

    if "is_relevant" in data.columns:
        print("\nPhân phối is_relevant:")
        print(data["is_relevant"].value_counts(dropna=False).sort_index())

    if "manual_label" in data.columns:
        print("\nPhân phối manual_label:")
        print(data["manual_label"].value_counts(dropna=False).sort_index())

    print("\nVí dụ 5 dòng sau clean:")
    preview_cols = [col for col in ["id", "topic", "text", "clean_text"] if col in data.columns]
    print(data[preview_cols].head(5).to_string(index=False))


# ============================================================
# 6. MAIN PIPELINE
# ============================================================

def main():
    # -----------------------------
    # Đọc full dataset
    # -----------------------------
    df = read_table(input_full_path)

    required_cols = ["topic", "text"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Full dataset thiếu cột bắt buộc: {col}")

    print("Đã đọc full dataset:", input_full_path)
    print("Số dòng full:", len(df))

    # -----------------------------
    # Clean full dataset
    # -----------------------------
    df = df.copy()
    df["clean_text"] = df.apply(
        lambda row: clean_comment(row["text"], row["topic"]),
        axis=1
    )

    df["clean_text"] = df["clean_text"].fillna("").astype(str)

    quality_check(df, name="full dataset")

    df.to_csv(output_full_path, index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu full dataset đã clean tại: {output_full_path}")

    # -----------------------------
    # Clean gold set trực tiếp từ text + topic của gold
    # Không merge bằng id.
    # -----------------------------
    gold_file = Path(gold_path)

    if not gold_file.exists():
        print(f"\nKhông tìm thấy gold set: {gold_path}. Bỏ qua bước clean gold.")
        return

    gold = read_table(gold_path)

    for col in required_cols:
        if col not in gold.columns:
            raise ValueError(f"Gold set thiếu cột bắt buộc: {col}")

    print("\nĐã đọc gold set:", gold_path)
    print("Số dòng gold:", len(gold))

    gold = gold.copy()
    gold["clean_text"] = gold.apply(
        lambda row: clean_comment(row["text"], row["topic"]),
        axis=1
    )

    gold["clean_text"] = gold["clean_text"].fillna("").astype(str)

    quality_check(gold, name="gold set")

    # -----------------------------
    # Kiểm tra gold match với full bằng khóa mạnh.
    # Chỉ print để audit, không tạo file phụ.
    # -----------------------------
    match_report = match_gold_to_full(df, gold)

    matched_count = int((match_report["matched_full_count"] > 0).sum())
    match_rate = matched_count / len(gold) if len(gold) > 0 else 0

    print("\n========== GOLD-FULL MATCH CHECK ==========")
    print("Số gold rows:", len(gold))
    print("Số gold rows match được:", matched_count)
    print("Gold row matched rate:", round(match_rate, 4))

    print("\nGold rows match theo topic:")
    print(
        match_report[match_report["matched_full_count"] > 0]["gold_topic"]
        .value_counts()
        .sort_index()
    )

    print("\nMatch method distribution:")
    print(match_report["match_method"].value_counts(dropna=False))

    unmatched = match_report[match_report["matched_full_count"] == 0]
    if len(unmatched) > 0:
        print("\nCảnh báo: còn gold rows chưa match được:")
        print(unmatched.head(10).to_string(index=False))
    else:
        print("\nTất cả gold rows đều match được với full dataset.")

    gold.to_csv(output_gold_path, index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu gold set đã clean tại: {output_gold_path}")


if __name__ == "__main__":
    main()


Đã đọc full dataset: labeled_data.csv
Số dòng full: 10718
========== QUALITY CHECK: full dataset ==========
Tổng số dòng: 10718
Số clean_text trống: 1
Số clean_text còn URL gốc: 0
Số clean_text chưa NFC: 0
Số clean_text còn ký tự chữ lặp >= 3: 0

Phân phối topic:
topic
arsenal           2555
iphone17series    2418
ronaldo            918
s25series         2122
vinfast_vf3       2705
Name: count, dtype: int64

Phân phối is_relevant:
is_relevant
0    2393
1    8325
Name: count, dtype: int64

Phân phối manual_label:
manual_label
0    2313
1    2563
2    5842
Name: count, dtype: int64

Ví dụ 5 dòng sau clean:
 id   topic                                                                          text                                                                                                  clean_text
  1 arsenal                                               GÁY TO LÊN ANH EM PHÁO THỦ ƠIII                                                              gáy to lên anh em arsenal arsenal pháo 

C:\Users\nguye\AppData\Local\Temp\ipykernel_30616\3068410614.py:463: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data["clean_text"].str.contains(r"([a-zA-Zà-ỹÀ-Ỹ])\1{2,}", regex=True, na=False).sum())
C:\Users\nguye\AppData\Local\Temp\ipykernel_30616\3068410614.py:463: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data["clean_text"].str.contains(r"([a-zA-Zà-ỹÀ-Ỹ])\1{2,}", regex=True, na=False).sum())



========== GOLD-FULL MATCH CHECK ==========
Số gold rows: 500
Số gold rows match được: 500
Gold row matched rate: 1.0

Gold rows match theo topic:
gold_topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Match method distribution:
match_method
topic+video_id+created_at+text    500
Name: count, dtype: int64

Tất cả gold rows đều match được với full dataset.

Đã lưu gold set đã clean tại: human_gold_500_cleaned_final.csv
